In [3]:
import string
import numpy as np
import os
from pickle import dump, load
from PIL import Image
import tensorflow as tf
import matplotlib.pyplot as plt
from keras.applications.xception import Xception, preprocess_input
from keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical, get_file
from keras.layers import add
from keras.models import Model, load_model
from keras.layers import Input, Dense, LSTM, Embedding, Dropout

In [4]:
from tqdm.notebook import tqdm as tqdm
tqdm().pandas()

0it [00:00, ?it/s]

In [5]:
def load_doc(filename):
  file = open(filename, 'r')
  text = file.read()
  file.close()
  return text

In [ ]:
def all_img_captions(filename):
  file = load_doc(filename)
  captions = file.split('\n')
  descriptions = {}
  for caption in captions[:-1]:
    img, caption = caption.split('\t')
    if img[:-2] not in descriptions:
      descriptions[img[:-2]] = [caption]
    else:
      descriptions[img[:-2]].append(caption)
  return descriptions

In [ ]:
def cleaning_text(captions):
  table = str.maketrans('', '', string.punctuation)
  for img, caps in captions.items():
    for i, img_caption in enumerate(caps):
      img_caption.replace("-","")
      desc = img_caption.split()
      desc = [word.lower() for word in desc]
      desc = [word.translate(table) for word in desc]
      desc = [word for word in desc if(len(word)>1)]
      desc = [word for word in desc if(word.isalpha())]

      img_caption = ' '.join(desc)
      captions[img][i]=img_caption
  return captions

In [ ]:
def text_vocabulary(descriptions):
  vocab = set()
  for key in descriptions.keys():
    [vocab.update(d.split()) for d in descriptions[key]]
  return vocab

In [ ]:
def save_descriptions( descriptions, filename):
  lines = list()
  for key, desc_list in descriptions.items():
    for desc in desc_list:
      lines.append(key + '\t' + desc)
  data = "\n".join(lines)
  file = open(filename, "w")
  file.write(data)
  file.close

In [6]:
dataset_text = "/kaggle/input/datasets/rajburnwal/flickr8k-text"
dataset_images = "/kaggle/input/datasets/rajburnwal/flickr8k-dataset/Flicker8k_Dataset"

In [ ]:
filename = dataset_text + "/Flickr8k.token.txt"
descriptions = all_img_captions(filename)
print("Length of descriptions = ", len(descriptions))

In [ ]:
clean_descriptions = cleaning_text(descriptions)
vocabulary = text_vocabulary(clean_descriptions)
print("Length of vocabulary = ", len(vocabulary))

In [ ]:
save_descriptions(clean_descriptions, "descriptions.txt")

In [45]:
def download_with_retry(url, filename, max_retries = 3):
  for attempt in range(max_retries):
    try:
      path = get_file(filename, url)
      return path
    except Exception as e:
      if attempt < max_retries - 1:
        raise e
      print(f"Download attempt failed")
      time.sleep(3)

In [46]:
weights_url = "https://storage.googleapis.com/tensorflow/keras-applications/xception/xception_weights_tf_dim_ordering_tf_kernels_notop.h5"
weights_path = download_with_retry(weights_url, "xception_weights.h5")
model = Xception(include_top=False, pooling="avg", weights=weights_path)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [42]:
def extract_features(directory):
  features = {}
  valid_images = ['.jpg','jpeg','.png']
  for img in tqdm(os.listdir(directory)):
    ext = os.path.splitext(img)[1].lower()
    if ext.lower() not in valid_images:
      continue
    filename = directory + "/" + img
    image = Image.open(filename)
    image = image.resize((299,299))
    image = np.expand_dims(image, axis=0)
    image = image/127.5

    feature = model.predict(image)
    features[img] = feature

  return features

In [ ]:
features = extract_features(dataset_imges)
dump(features, open("features.p", "wb"))

In [56]:
features = load(open("/kaggle/input/datasets/rajburnwal/generated-files/features.p", "rb"))

In [8]:
def load_photos(filename):
  file=load_doc(filename)
  photos = file.split("\n")[:-1]
  photos_present = [photo for photo in photos if os.path.exists(os.path.join(dataset_images, photo))]
  return photos_present

In [9]:
def load_clean_description(filename, photos):
  file = load_doc(filename)
  descriptions = {}
  for line in file.split("\n"):
    words = line.split()
    if len(words)<1:
      continue
    image, image_caption = words[0], words[1:]
    if image in photos:
      if image not in descriptions:
        descriptions[image] = []
      desc='<start>' + " ".join(image_caption) + ' <end>'
      descriptions[image].append(desc)
  return descriptions

In [10]:
def load_features(photos):
  all_features = load(open("/kaggle/input/datasets/rajburnwal/generated-files/features.p", "rb"))
  features = {k:all_features[k] for k in photos}
  return features

In [11]:
filename= dataset_text + '/' + 'Flickr_8k.trainImages.txt'
train_imgs = load_photos(filename)
val_images = load_photos('/kaggle/input/datasets/rajburnwal/flickr8k-text/Flickr_8k.devImages.txt')
train_descriptions = load_clean_description("/kaggle/input/datasets/rajburnwal/generated-files/descriptions.txt", train_imgs)
val_descriptions = load_clean_description(
    "/kaggle/input/datasets/rajburnwal/generated-files/descriptions.txt",
    val_images
)
train_features = load_features(train_imgs)

In [12]:
val_features = load_features(val_images)

In [13]:
print("Validation descriptions:", len(val_descriptions))

Validation descriptions: 1000


In [14]:
print("Training features:", len(train_features))
print("Validation features:", len(val_features))

print("Validation descriptions:", len(val_descriptions))
print("Missing validation features:",
      len(set(val_descriptions.keys()) - set(val_features.keys())))

Training features: 6000
Validation features: 1000
Validation descriptions: 1000
Missing validation features: 0


In [15]:
def dict_to_list(descriptions):
  all_desc = []
  for key in descriptions.keys():
    [all_desc.append(d) for d in descriptions[key]]
  return all_desc

In [ ]:
def create_tokenizer(descriptions):
  desc_list = dict_to_list(descriptions)
  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(desc_list)
  return tokenizer

In [ ]:
tokenizer = create_tokenizer(train_descriptions)

In [ ]:
dump(tokenizer, open('tokenizer.p', 'wb'))

In [16]:
tokenizer=load(open("/kaggle/input/datasets/rajburnwal/generated-files/tokenizer.p","rb"))

In [17]:
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

7577


In [18]:
def max_length(descriptions):
    desc_list = dict_to_list(descriptions)
    return max(len(d.split()) for d in desc_list)

In [19]:
max_length = max_length(train_descriptions)
print(max_length)

33


In [20]:
def data_generator(descriptions, features, tokenizer, max_length):

    def generator():
        for key, description_list in descriptions.items():
            feature = features[key][0]

            X1, X2, y = create_sequences(
                tokenizer,
                max_length,
                description_list,
                feature
            )

            for i in range(len(X1)):
                yield (
                    {
                        'input_1': X1[i],
                        'input_2': X2[i]
                    },
                    y[i]
                )

    output_signature = (
        {
            'input_1': tf.TensorSpec(
                shape=(2048,),
                dtype=tf.float32
            ),
            'input_2': tf.TensorSpec(
                shape=(max_length,),
                dtype=tf.int32
            )
        },
        tf.TensorSpec(
            shape=(),
            dtype=tf.int32
        )
    )

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=output_signature
    )

    return dataset.batch(32).prefetch(tf.data.AUTOTUNE).repeat()

In [21]:
def create_sequences(tokenizer, max_length, desc_list, feature):
    X1, X2, y = [], [], []

    for desc in desc_list:
        seq = tokenizer.texts_to_sequences([desc])[0]

        for i in range(1, len(seq)):
            in_seq = seq[:i]
            in_seq = pad_sequences(
                [in_seq],
                maxlen=max_length,
                padding='post'
            )[0]

            out_word = seq[i]

            X1.append(feature)
            X2.append(in_seq)
            y.append(out_word)

    return (
        np.array(X1, dtype=np.float32),
        np.array(X2, dtype=np.int32),
        np.array(y, dtype=np.int32)
    )

In [22]:
def define_model(vocab_size, max_length):

    # features from the CNN model squeezed from 2048 to 256 nodes
    inputs1 = Input(shape=(2048,), name='input_1')
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    # LSTM sequence model
    inputs2 = Input(shape=(max_length,), name='input_2')
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256)(se2)

    # Merging both models
    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    # tie it together [image, seq] [word]
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

    # summarize model
    print(model.summary())

    return model

In [23]:
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

7577


In [24]:
def max_length(descriptions):
    desc_list = dict_to_list(descriptions)
    return max(len(d.split()) for d in desc_list)

In [25]:
max_length = max_length(train_descriptions)
print(max_length)

33


In [55]:
first_key = next(iter(train_descriptions))

print(first_key)
print(len(train_descriptions[first_key]))
print([len(d.split()) for d in train_descriptions[first_key]])
print("Vocabulary:", vocab_size)
print("Max length:", max_length)

1000268201_693b08cb0e.jpg
5
[15, 6, 7, 9, 10]
Vocabulary: 7577
Max length: 33


In [26]:
model = define_model(vocab_size, max_length)
epochs = 10

I0000 00:00:1786787214.829990      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786787214.836010      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_2             │ (None, 33)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_1             │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 33, 256)   │  1,939,712 │ input_2[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ input_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 33, 256)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 33)        │          0 │ input_2[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 256)       │    525,312 │ dropout_1[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256)       │          0 │ dense[0][0],      │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 7577)      │  1,947,289 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,002,649 (19.08 MB)

 Trainable params: 5,002,649 (19.08 MB)

 Non-trainable params: 0 (0.00 B)

None


In [27]:
import math
def get_steps_per_epoch(train_descriptions):
    total_sequences = 0
    for img_captions in train_descriptions.values():
        for caption in img_captions:
            words = caption.split()
            total_sequences += len(words) - 1
    # Ensure at least 1 step, even if sequences < batch_size
    return math.ceil(total_sequences / 32)

In [28]:
steps = get_steps_per_epoch(train_descriptions)
val_steps = get_steps_per_epoch(val_descriptions)

# making a directory models to save our models
os.mkdir("models2")

dataset = data_generator(
    train_descriptions,
    train_features,
    tokenizer,
    max_length
)
val_dataset = data_generator(
    val_descriptions,
    val_features,
    tokenizer,
    max_length
)

In [31]:
from tensorflow.keras.callbacks import ModelCheckpoint

In [32]:
checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

In [34]:
model.fit(
    dataset,
    epochs=epochs,
    steps_per_epoch=steps,
    validation_data=val_dataset,
    validation_steps=val_steps,
    callbacks=[checkpoint],
    verbose=1
)


Epoch 1/10
8636/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0622
Epoch 1: val_loss improved from None to 4.07679, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 131s 15ms/step - loss: 3.0829 - val_loss: 4.0768
Epoch 2/10
8637/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0581
Epoch 2: val_loss did not improve from 4.07679
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 129s 15ms/step - loss: 3.0635 - val_loss: 4.0968
Epoch 3/10
8636/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0756
Epoch 3: val_loss did not improve from 4.07679
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 130s 15ms/step - loss: 3.0556 - val_loss: 4.1576
Epoch 4/10
8636/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0710
Epoch 4: val_loss improved from 4.07679 to 4.07373, saving model to best_model.h5



Epoch 4: finished saving model to best_model.h5
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 130s 15ms/step - loss: 3.0558 - val_loss: 4.0737
Epoch 5/10
8635/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0404
Epoch 5: val_loss did not improve from 4.07373
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 132s 15ms/step - loss: 3.0420 - val_loss: 4.1431
Epoch 6/10
8637/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0557
Epoch 6: val_loss did not improve from 4.07373
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 130s 15ms/step - loss: 3.0415 - val_loss: 4.1785
Epoch 7/10
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0470
Epoch 7: val_loss did not improve from 4.07373
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 130s 15ms/step - loss: 3.0403 - val_loss: 4.2601
Epoch 8/10
8636/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.0395
Epoch 8: val_loss did not improve from 4.07373
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 132s 15ms/step - loss: 3.0452 - val_loss: 4.4015
Epoch 9/10
8638/8638 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.9949
Epoch 9: val_

In [36]:
import os
import shutil

os.makedirs("models2", exist_ok=True)

shutil.copy(
    "best_model.h5",
    "models2/best_model.h5"
)

'models2/best_model.h5'

In [37]:
print(os.path.exists("models2/best_model.h5"))

True


In [38]:
import os

print(os.path.abspath("best_model.h5"))

/kaggle/working/best_model.h5


In [40]:
model = define_model(vocab_size, max_length)
model.load_weights("best_model.h5")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_2             │ (None, 33)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_1             │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 33, 256)   │  1,939,712 │ input_2[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 2048)      │          0 │ input_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 33, 256)   │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 33)        │          0 │ input_2[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    524,544 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 256)       │    525,312 │ dropout_3[0][0],  │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256)       │          0 │ dense_3[0][0],    │
│                     │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 7577)      │  1,947,289 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,002,649 (19.08 MB)

 Trainable params: 5,002,649 (19.08 MB)

 Non-trainable params: 0 (0.00 B)

None


In [61]:
import numpy as np
from pickle import load
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score


# ============================================================
# 1. LOAD TOKENIZER
# ============================================================

tokenizer = load(
    open(
        "/kaggle/input/datasets/rajburnwal/generated-files/tokenizer.p",
        "rb"
    )
)

vocab_size = len(tokenizer.word_index) + 1
max_length = 33


# ============================================================
# 2. LOAD TEST IMAGE NAMES
# ============================================================

TEST_IMAGES_FILE = (
    "/kaggle/input/datasets/rajburnwal/flickr8k-text/"
    "Flickr_8k.testImages.txt"
)

CAPTIONS_FILE = (
    "/kaggle/input/datasets/rajburnwal/generated-files/"
    "descriptions.txt"
)

with open(TEST_IMAGES_FILE, "r") as f:
    test_images = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Number of test images:", len(test_images))


# ============================================================
# 3. LOAD PRECOMPUTED FEATURES
# ============================================================

test_features = load_features(test_images)

print("Test features:", len(test_features))

missing_features = [
    image for image in test_images
    if image not in test_features
]

print("Missing test features:", len(missing_features))

if len(missing_features) > 0:
    raise ValueError("Some test images do not have precomputed features.")


# ============================================================
# 4. LOAD GROUND-TRUTH CAPTIONS
# ============================================================

def load_test_descriptions(filename, test_images):

    test_set = set(test_images)
    descriptions = {}

    with open(filename, "r") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            words = line.split()

            image = words[0]
            caption_words = words[1:]

            if image not in test_set:
                continue

            caption = " ".join(caption_words)

            if image not in descriptions:
                descriptions[image] = []

            descriptions[image].append(caption)

    return descriptions


test_descriptions = load_test_descriptions(
    CAPTIONS_FILE,
    test_images
)

print(
    "Test images with captions:",
    len(test_descriptions)
)


# ============================================================
# 5. CREATE REVERSE WORD LOOKUP
# ============================================================

id_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}


# ============================================================
# 6. BATCHED GREEDY CAPTION GENERATION
# ============================================================

def generate_captions_batch(
    model,
    tokenizer,
    test_images,
    test_features,
    max_length
):

    # --------------------------------------------------------
    # Stack all precomputed image features
    # Shape: (number_of_images, 2048)
    # --------------------------------------------------------

    photos = np.stack([
        test_features[image][0]
        for image in test_images
    ]).astype(np.float32)


    # --------------------------------------------------------
    # Start every caption with "start"
    # --------------------------------------------------------

    start_id = tokenizer.word_index["start"]

    end_id = tokenizer.word_index.get("end")

    sequences = [
        [start_id]
        for _ in test_images
    ]


    # Keep track of images that have not generated "end"
    active_indices = list(range(len(test_images)))


    # --------------------------------------------------------
    # Greedy decoding
    #
    # One model call processes ALL currently active images.
    # This is much faster than calling model.predict()
    # separately for every image and every word.
    # --------------------------------------------------------

    for step in range(max_length - 1):

        if not active_indices:
            break

        active_sequences = [
            sequences[i]
            for i in active_indices
        ]

        padded_sequences = pad_sequences(
            active_sequences,
            maxlen=max_length,
            padding="post"
        ).astype(np.int32)


        active_photos = photos[active_indices]


        # Direct TensorFlow model call
        predictions = model(
            [
                active_photos,
                padded_sequences
            ],
            training=False
        )


        # Highest-probability next word
        next_words = np.argmax(
            predictions.numpy(),
            axis=1
        )


        new_active_indices = []


        for j, image_index in enumerate(active_indices):

            next_word_id = int(next_words[j])

            sequences[image_index].append(
                next_word_id
            )


            # Stop generating for this image
            # once "end" is produced.
            if (
                end_id is None
                or next_word_id != end_id
            ):
                new_active_indices.append(
                    image_index
                )


        active_indices = new_active_indices


    # --------------------------------------------------------
    # Convert token IDs back to words
    # --------------------------------------------------------

    captions = []

    for sequence in sequences:

        words = []

        for token_id in sequence:

            word = id_to_word.get(
                int(token_id)
            )

            if word is not None:
                words.append(word)

        captions.append(
            " ".join(words)
        )

    return captions


# ============================================================
# 7. GENERATE ALL TEST CAPTIONS
# ============================================================

print("\nGenerating captions...")

predictions = generate_captions_batch(
    model,
    tokenizer,
    test_images,
    test_features,
    max_length
)

print("Caption generation complete.")


# ============================================================
# 8. PREPARE REFERENCES AND HYPOTHESES
# ============================================================

references = []
hypotheses = []


for image_name, prediction in zip(
    test_images,
    predictions
):

    if image_name not in test_descriptions:
        continue


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    prediction_words = prediction.split()

    if (
        prediction_words
        and prediction_words[0] == "start"
    ):
        prediction_words = prediction_words[1:]

    if (
        prediction_words
        and prediction_words[-1] == "end"
    ):
        prediction_words = prediction_words[:-1]


    # --------------------------------------------------------
    # Ground-truth captions
    # --------------------------------------------------------

    reference_words = []

    for caption in test_descriptions[image_name]:

        words = caption.split()

        if words and words[0] == "start":
            words = words[1:]

        if words and words[-1] == "end":
            words = words[:-1]

        reference_words.append(words)


    references.append(reference_words)
    hypotheses.append(prediction_words)


# ============================================================
# 9. BLEU SCORES
# ============================================================

smoothie = SmoothingFunction().method1


bleu1 = corpus_bleu(
    references,
    hypotheses,
    weights=(1, 0, 0, 0),
    smoothing_function=smoothie
)


bleu2 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smoothie
)


bleu3 = corpus_bleu(
    references,
    hypotheses,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smoothie
)


bleu4 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smoothie
)


# ============================================================
# 10. METEOR
# ============================================================

meteor_scores = []

for refs, hypothesis in zip(
    references,
    hypotheses
):

    score = meteor_score(
        refs,
        hypothesis
    )

    meteor_scores.append(score)


meteor = np.mean(meteor_scores)


# ============================================================
# 11. RESULTS
# ============================================================

print("\n==============================")
print("IMAGE CAPTIONING RESULTS")
print("==============================")

print(
    f"Test images evaluated : {len(hypotheses)}"
)

print(
    f"BLEU-1                : {bleu1:.4f}"
)

print(
    f"BLEU-2                : {bleu2:.4f}"
)

print(
    f"BLEU-3                : {bleu3:.4f}"
)

print(
    f"BLEU-4                : {bleu4:.4f}"
)

print(
    f"METEOR                : {meteor:.4f}"
)

print("==============================")

Number of test images: 1000
Test features: 1000
Missing test features: 0
Test images with captions: 1000

Generating captions...
Caption generation complete.

IMAGE CAPTIONING RESULTS
Test images evaluated : 1000
BLEU-1                : 0.3879
BLEU-2                : 0.1955
BLEU-3                : 0.1024
BLEU-4                : 0.0520
METEOR                : 0.2562
